In [ ]:
%pip install kagglehub

In [2]:
from pathlib import Path
import kagglehub
import pandas as pd

In [ ]:
path = Path(kagglehub.dataset_download("vagifa/ethereum-frauddetection-dataset", output_dir=(Path.cwd() / "data"), force_download=True))

print("저장 위치:", path)

In [4]:
# 다운받은 데이터셋 삭제
# import shutil
# shutil.rmtree(path)

In [5]:
# df = pd.read_csv(path / "transaction_dataset.csv")
df = pd.read_csv("data/transaction_dataset.csv")
print(df.shape)
df.columns = df.columns.str.strip()
df.keys()

(9841, 51)


Index(['Unnamed: 0', 'Index', 'Address', 'FLAG', 'Avg min between sent tnx',
       'Avg min between received tnx',
       'Time Diff between first and last (Mins)', 'Sent tnx', 'Received Tnx',
       'Number of Created Contracts', 'Unique Received From Addresses',
       'Unique Sent To Addresses', 'min value received', 'max value received',
       'avg val received', 'min val sent', 'max val sent', 'avg val sent',
       'min value sent to contract', 'max val sent to contract',
       'avg value sent to contract',
       'total transactions (including tnx to create contract',
       'total Ether sent', 'total ether received',
       'total ether sent contracts', 'total ether balance', 'Total ERC20 tnxs',
       'ERC20 total Ether received', 'ERC20 total ether sent',
       'ERC20 total Ether sent contract', 'ERC20 uniq sent addr',
       'ERC20 uniq rec addr', 'ERC20 uniq sent addr.1',
       'ERC20 uniq rec contract addr', 'ERC20 avg time between sent tnx',
       'ERC20 avg time be

In [6]:
# 결측 확인
n_rows, n_cols = df.shape

na_count = df.isna().sum()
na_ratio = (na_count / n_rows * 100)
na_summary = pd.DataFrame({
    "na_count": na_count,
    "na_ratio_%": na_ratio
})
res = na_summary[na_summary["na_count"] > 0].sort_values("na_count", ascending=False)
display(res)
print(len(res.index))
# ERC20 most sent token type: 보낸 ERC20 토큰 자체가 없어서 "가장 많이 보낸 토큰 종류"도 정의할 수 없는 경우
# 정확히 같은 829개 행에서 동시에 결측: 특정 fraud 주소군에 대해 ERC20 특징을 추출하지 못했거나 원천 데이터가 없었던 경우

,na_count,na_ratio_%
ERC20 most sent token type,2697,27.405751
ERC20_most_rec_token_type,871,8.850727
ERC20 total ether sent,829,8.423941
ERC20 total Ether received,829,8.423941
ERC20 uniq sent addr,829,8.423941
ERC20 uniq rec addr,829,8.423941
ERC20 uniq sent addr.1,829,8.423941
ERC20 total Ether sent contract,829,8.423941
Total ERC20 tnxs,829,8.423941
ERC20 avg time between sent tnx,829,8.423941


25


In [7]:
# 한 행에서 어떤 컬럼들이 한번에 결측인지
# GPT가 CSV 전체를 읽어보니 829개짜리는 전부 FLAG=1이라고 함 근데 결측 없는 FLAG=1도 있어서 csv 만든쪽에서 공격패턴 의도적으로 만들다 실수한듯
na_pat = df.apply(lambda row: tuple(row.index[row.isna()]), axis=1)
na_pat = na_pat[na_pat.map(len) > 0]
na_stat = na_pat.value_counts().rename_axis("na_cols").reset_index(name="count")
na_stat["n_na_cols"] = na_stat["na_cols"].map(len)
with pd.option_context("display.max_colwidth", 60): display(na_stat.sort_values("n_na_cols", ascending=False))

,na_cols,count,n_na_cols
1,"(Total ERC20 tnxs, ERC20 total Ether received, ERC20 tot...",829,25
3,"(ERC20 most sent token type, ERC20_most_rec_token_type)",19,2
0,"(ERC20 most sent token type,)",1849,1
2,"(ERC20_most_rec_token_type,)",23,1


In [8]:
n_rows, n_cols = df.shape
print(f"전체 shape: {df.shape}")

na_count = df.isna().sum()
na_ratio = (na_count / n_rows * 100).round(2)

na_summary = pd.DataFrame({
    "nan_count": na_count,
    "nan_ratio_%": na_ratio
}).sort_values(["nan_count", "nan_ratio_%"], ascending=[False, False])

na_cols = na_summary[na_summary["nan_count"] > 0]
full_cols = na_summary[na_summary["nan_count"] == 0]

rows_with_any_nan = df.isna().any(axis=1).sum()
print(f"\n결측이 1개 이상 있는 행 수: {rows_with_any_nan} / {n_rows} ({rows_with_any_nan / n_rows * 100:.2f}%)")

res = df[df.isna().any(axis=1)].sample(3).T
res = res.map(lambda x: x if pd.isna(x) else str(x)[:15])
with pd.option_context("display.max_colwidth", 15): display(res.style.highlight_null(props="background-color: black; color: #e6e6e6;"))

전체 shape: (9841, 51)

결측이 1개 이상 있는 행 수: 2720 / 9841 (27.64%)


,9711,4525,4872
Unnamed: 0,9711,4525,4872
Index,2050,1593,1940
Address,0xf0e28a5485abe,0x7879a595f01c5,0x810faab062300
FLAG,1,0,0
Avg min between sent tnx,8334.04,0.0,39.61
Avg min between received tnx,5.3,11343.75,2317.83
Time Diff between first and last (Mins),16678.68,136124.95,982974.33
Sent tnx,2,0,415
Received Tnx,2,12,417
Number of Created Contracts,0,1,0


In [9]:
# 2697개가 결측이던 ERC20 most sent token type 컬럼 진짜 결측인지 이유가 있는건지 논리적 확인
tok = 'ERC20 most sent token type'
sent_cols = ['ERC20 total ether sent', 'ERC20 uniq sent addr', 'ERC20 uniq sent token name']

base = df[sent_cols].fillna(0)

true_none = df[tok].isna() & base.eq(0).all(axis=1)      # 보낸 ERC20 자체가 없어서 토큰타입도 없음
real_missing = df[tok].isna() & base.gt(0).any(axis=1)   # 보낸 기록은 있는데 type만 비어 있음

print('진짜 없음:', true_none.sum())
print('진짜 누락 의심:', real_missing.sum())

df.loc[real_missing, [tok] + sent_cols].head(20)

진짜 없음: 2685
진짜 누락 의심: 12


,ERC20 most sent token type,ERC20 total ether sent,ERC20 uniq sent addr,ERC20 uniq sent token name
15,NaN,5.000000e-06,3.0,1.0
33,NaN,1.000000e-11,1.0,1.0
80,NaN,9.013333e-01,4.0,2.0
675,NaN,2.100000e-01,2.0,2.0
936,NaN,2.000000e+00,2.0,2.0
1049,NaN,2.508970e+03,4.0,4.0
2328,NaN,7.000000e+00,2.0,2.0
4642,NaN,7.798942e+01,2.0,2.0
5360,NaN,1.000000e-01,1.0,1.0
6387,NaN,4.000000e+00,2.0,2.0


In [ ]:
sorted(df["ERC20 most sent token type"].unique())

In [19]:
# GPT가 골라준 이더리움에서 입력받을 수 있는 컬럼들
cols = [
    "FLAG",
    "Avg min between sent tnx", "Avg min between received tnx", "Time Diff between first and last (Mins)", "Sent tnx", "Received Tnx", 
    "Number of Created Contracts", "Unique Received From Addresses", "Unique Sent To Addresses", "min value received", "max value received", "avg val received", 
    "min val sent", "max val sent", "avg val sent", "min value sent to contract", "max val sent to contract", "avg value sent to contract", 
    "total transactions (including tnx to create contract", "total Ether sent", "total ether received", "total ether sent contracts", "total ether balance", 
    "Total ERC20 tnxs", "ERC20 total Ether received", "ERC20 total ether sent", "ERC20 total Ether sent contract", "ERC20 uniq sent addr", 
    "ERC20 uniq rec addr", "ERC20 uniq rec contract addr", "ERC20 avg time between sent tnx", "ERC20 avg time between rec tnx", "ERC20 avg time between contract tnx", 
    "ERC20 min val rec", "ERC20 max val rec", "ERC20 avg val rec", "ERC20 min val sent", "ERC20 max val sent", "ERC20 avg val sent", 
    "ERC20 min val sent contract", "ERC20 max val sent contract", "ERC20 avg val sent contract"
]
# 파생 컬럼
made_cols = [
    "has_erc20_activity", #Total ERC20 tnxs > 0 이면 1 아니면 0, 토큰 활동 여부
    "sent_received_ratio", # total Ether sent / (total ether received + 1e-9), 송금과 수신의 비율
    "unique_counterparty_ratio" # Unique Sent To Addresses / (Sent tnx + 1e-9), 송금 거래에서 고유 상대방 주소의 비율
]

all_cols = cols + made_cols

In [ ]:
# NaN 전처리

# 1. ERC20 활동 없음으로 볼 수 있는 수치형 컬럼
erc20_num_cols = [
    "Total ERC20 tnxs", "ERC20 total Ether received", "ERC20 total ether sent",
    "ERC20 total Ether sent contract", "ERC20 uniq sent addr", "ERC20 uniq rec addr",
    "ERC20 uniq rec contract addr", "ERC20 avg time between sent tnx",
    "ERC20 avg time between rec tnx", "ERC20 avg time between contract tnx",
    "ERC20 min val rec", "ERC20 max val rec", "ERC20 avg val rec",
    "ERC20 min val sent", "ERC20 max val sent", "ERC20 avg val sent",
    "ERC20 min val sent contract", "ERC20 max val sent contract",
    "ERC20 avg val sent contract"
]

df[erc20_num_cols] = df[erc20_num_cols].fillna(0)

# # 2. 토큰 타입 컬럼 (근데 모델학습에 안씀)
# token_cols = ["ERC20 most sent token type", "ERC20_most_rec_token_type"]
# df[token_cols] = df[token_cols].fillna("none")

# # 3. 일반 수치형 컬럼 중 애매한 누락은 중앙값 (근데 모델학습에 안씀)
# other_num_cols = [c for c in cols if c in df.columns and df[c].dtype != "O" and c not in erc20_num_cols]
# df[other_num_cols] = df[other_num_cols].fillna(df[other_num_cols].median())

In [21]:
# 파생 컬럼 등록
if "has_erc20_activity" not in df.columns:
    df["has_erc20_activity"] = (df["Total ERC20 tnxs"].fillna(0) > 0).astype(int)

if "sent_received_ratio" not in df.columns:
    df["sent_received_ratio"] = (df["total Ether sent"].fillna(0) / (df["total ether received"].fillna(0) + 1e-9))

if "unique_counterparty_ratio" not in df.columns:
    df["unique_counterparty_ratio"] = (df["Unique Sent To Addresses"].fillna(0) / (df["Sent tnx"].fillna(0) + 1e-9))

print([i for i in made_cols if i not in df.columns])

[]


In [ ]:
# 결측 없는지 확인
df2 = df[all_cols]
df2.shape, df2.dropna().shape

((9841, 45), (9841, 45))

In [ ]:
# 저장
FINAL_DF = df[all_cols]
FINAL_DF.to_csv("data/dataset.csv", index=False, encoding="utf-8-sig")

In [ ]:
# 원본 다시 불러오기
df = pd.read_csv("data/transaction_dataset.csv")
df.columns = df.columns.str.strip()